# Complete the AI READI reviewer checks

This notebook supports Comments 1–5 on the **CGM-independent revised run**. It recovers saved model specifications, checks provenance, calculates age-group errors and supplies a three-seed SHAP check. An optional, separate analysis refits the two SDH-containing groups without education.

The original analysis remains exploratory because test results were examined earlier. This notebook does not change that status. It writes new files in a timestamped `reviewer_completion` folder, keeps participant data local and preserves existing models and reports.

**Start with a fresh Jupyter kernel.** Open this notebook from the extracted toolkit folder. No package installation or wearable extraction is performed. Use the environment compatible with the saved models. A version mismatch is a reason to restore that environment, not to reinstall the latest packages.


## 1 Locate the existing run

The relocated run path is prefilled from this project. The toolkit includes the reviewed source modules; their hashes must agree with the original training lock before use. If the executed source differs, supply its actual archived folder instead of changing the lock.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display

TOOLKIT_DIR = Path.cwd()
RUN_DIR = Path(r"D:\POSTDOC\YSN\AimAhead\AI_READI_Analysis\Redo3\hba1c_no_cgm_v1\20260927_192923_602247")
DATASET_DIR = Path(r"D:\POSTDOC\YSN\AimAhead\DATA\aireadi\11a60cdd-6588-4f77-80d8-4e2718613172\dataset")
CODE_DIR = TOOLKIT_DIR / "reviewed_source"

if not (TOOLKIT_DIR / "reviewer_completion.py").is_file():
    raise FileNotFoundError("Open this notebook from the extracted AI_READI_Reviewer_Completion_Toolkit folder, "
                            "or set TOOLKIT_DIR to that folder. It must contain reviewer_completion.py.")
sys.path.insert(0, str(TOOLKIT_DIR))
import reviewer_completion as revision
CTX = revision.setup(RUN_DIR, CODE_DIR)


## 2 Recover parameters and encoded dimensions

This reads fitted objects and reproduces each preprocessor's output width. It performs no fitting. It checks selected parameters against the saved inner search record and exports both the eight-group dimension summary and the 24-learner specification.


In [ ]:
specifications = revision.model_specifications(CTX)
display(specifications)


## 3 Complete the documentary provenance

Fill a value only when the original release/download record or code archive establishes it. Do not use the folder UUID as a release DOI, a file's modification time as a download date, or this notebook's package versions as the fitting environment. The recorded environment was created before preparation; fitting-time continuity still needs evidence.


In [ ]:
AUTHOR_RECORD = {
    "dataset_release_identifier": None,
    "dataset_release_doi": None,
    "acquisition_date": None,
    "acquisition_evidence": None,
    "training_environment_evidence": None,
    "public_code_repository": None,
    "immutable_commit_or_doi": None,
}
revision.provenance_inputs(CTX, DATASET_DIR, AUTHOR_RECORD)


## 4 Trace education to its original source

The code finds candidate source labels and concepts and searches code-only notebook cells for the derivation. It does not equate a candidate text match with verified coding. Candidate details and snippets remain in `LOCAL_ONLY`; inspect them on your computer.

Verification requires the source question/concept, units or response coding, missing-code rules, staging derivation and record-level reconciliation. If these cannot be established, use the separate exclusion analysis in Section 7. The original field name is insufficient evidence of years of schooling.


In [ ]:
SOURCE_CODE_FILES = [
    Path(r"C:\Users\elham\AIMAHEAD Anslysis Redo3.ipynb"),
    Path(r"D:\POSTDOC\YSN\AimAhead\AI_READI_Analysis\paper_finalization_v1\saved_source_recovery\AI_READI_original_analysis_CODE_ONLY.txt"),
]
education_candidates = revision.education_audit(CTX, DATASET_DIR, SOURCE_CODE_FILES)
print("Candidate mapping and code-snippet files are in:", CTX["local"])
# Display locally if needed; these candidates are not a verified final mapping.
# display(education_candidates)


## 5 Describe errors by age

Ages 40–54, 55–64 and ≥65 are evaluated separately in validation and test. The output includes MAE, bias, group sizes, observed HbA1c means and conditional bootstrap intervals. These are additional descriptive analyses chosen after test exposure. A small-group flag indicates limited precision; it does not remove participants. No demographic-fairness conclusion follows from these summaries.


In [ ]:
age_results = revision.age_strata(CTX)
display(age_results.loc[age_results["model"].eq("CWDS__ensemble")])


## 6 Check SHAP rankings across three seeds

This uses all 257 eligible validation participants and the same original 32 training background records. It repeats the original 128-path estimator using three new, recorded seeds. It can take approximately three times the original explanation runtime. Progress is printed.

The original `SHAP_metadata.json` and `background_ids_LOCAL_ONLY.csv` are required to test the published figure. The notebook does not silently substitute a different background. The resulting rank correlations and top-10 overlaps concern permutation-sampling variation conditional on that background and model; they do not establish causal effects or refitting stability. Original SHAP outputs are preserved.


In [ ]:
shap_stability_results = revision.shap_stability(CTX)
display(shap_stability_results)


## 7 Education exclusion if source coding remains unverified

**This section fits new models.** It refits only WDS and CWDS without education using the same eligible training participants and nested partitions. It preserves the original models and evaluates the new fits separately as a post-test sensitivity analysis. The six groups without education are reused unchanged.

If source-level verification is complete, this alternative is not required to address Comment 2. If verification fails, change the switch below to `True` and run it. Leaving it `False` does not resolve the education comment. This may take time because tuning and weight estimation are repeated.

If these models replace the reported SDH specification, their counts are 42 and 47 original variables. Update affected tables and regenerate calibration and SHAP figures from the new models. The old 48-variable Figure 5 cannot explain the new model.


In [ ]:
RUN_EDUCATION_EXCLUSION = False

if RUN_EDUCATION_EXCLUSION:
    education_sensitivity = revision.education_exclusion(CTX)
    display(education_sensitivity)
else:
    print("Education-exclusion fitting was not run. Source verification or this analysis remains required.")


## 8 Export the review results

The archive contains only the explicit aggregate and metadata outputs. Participant tables, background identifiers, model files and source snippets are excluded. Its manifest lists any outputs that were not completed. Review this archive under your applicable data-use conditions before sharing it with the manuscript.

The actual release/download evidence and a genuine public code citation may still need to be supplied separately. No code repository is published automatically.


In [ ]:
RESULTS_ARCHIVE = revision.export_review(CTX)
print(RESULTS_ARCHIVE)
